In [35]:
import pandas as pd
import numpy as np

train_df = pd.read_parquet("../data/train.parquet")
val_df = pd.read_parquet("../data/validation.parquet")
test_df = pd.read_parquet("../data/test.parquet")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (58516, 45)
Validation: (19506, 45)
Testing: (19506, 45)


In [36]:

# Prepare the features required by the M5 preprocessing pipeline

for df in [train_df, val_df, test_df]:
    df["categories_count"] = df["categories"].apply(len)
    df["genres_count"] = df["genres"].apply(len)

    df["release_month"] = (
        df["release_date"].dt.month.astype(str)
    )

print("Feature preparation completed!")

print("\nNew features:")
print(train_df[[
    "categories_count",
    "genres_count",
    "release_month"
]].head())


Feature preparation completed!

New features:
   categories_count  genres_count release_month
0                 3             3             5
1                 7             2             2
2                 7             2             6
3                 2             4             7
4                 5             2             8


In [37]:

from steam_preprocessing import create_preprocessor

preprocessor = create_preprocessor()

print("Preprocessing module imported successfully!")
print(preprocessor)


Preprocessing module imported successfully!
ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['price', 'required_age']),
                                ('boolean', 'passthrough',
                                 ['windows', 'mac', 'linux']),
                                ('counts', 'passthrough',
                                 ['categories_count', 'genres_count']),
                                ('month',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['release_month']),
                                ('text',
                                 Pipeline(steps=[('clean',
                                                  FunctionTransformer(func=<function clean_text_column at 0x000002A6339631A0>)),
                                                 ('tfidf',
                                                  TfidfVectorizer(max_features=5000,
                                        

In [38]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import f1_score

In [39]:

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import f1_score

from steam_preprocessing import create_preprocessor

# Create the M5 preprocessing setup without running M5
preprocessor = create_preprocessor()

# Create the Decision Tree using the M5 preprocessing
tree_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

# Train the model
tree_model.fit(train_df, train_df["hit"])

# Predict validation data
tree_predictions = tree_model.predict(val_df)

# Calculate F1-score
tree_f1 = f1_score(val_df["hit"], tree_predictions)

print("Decision Tree Results")
print("---------------------")
print("Validation F1-score:", round(tree_f1, 3))


Decision Tree Results
---------------------
Validation F1-score: 0.26


In [40]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline

# Different tree depths to compare
depths = [2, 3, 4, 8, None]

results = []

for depth in depths:

    model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", DecisionTreeClassifier(
            max_depth=depth,
            random_state=42
        ))
    ])

    # Train the model
    model.fit(train_df, train_df["hit"])

    # Predict validation data
    predictions = model.predict(val_df)

    # Calculate F1-score
    score = f1_score(val_df["hit"], predictions)

    results.append({
        "Max Depth": depth if depth is not None else "Unlimited",
        "Validation F1": round(score, 3)
    })

# Display results
results_df = pd.DataFrame(results)

print(results_df)

   Max Depth  Validation F1
0          2          0.427
1          3          0.260
2          4          0.383
3          8          0.422
4  Unlimited          0.449


In [41]:
depths = [2, 3, 4, 8, None]

overfitting_results = []

for depth in depths:

    model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", DecisionTreeClassifier(
            max_depth=depth,
            random_state=42
        ))
    ])

    model.fit(train_df, train_df["hit"])

    # Training predictions
    train_predictions = model.predict(train_df)
    train_score = f1_score(train_df["hit"], train_predictions)

    # Validation predictions
    val_predictions = model.predict(val_df)
    val_score = f1_score(val_df["hit"], val_predictions)

    overfitting_results.append({
        "Max Depth": depth if depth is not None else "Unlimited",
        "Training F1": round(train_score, 3),
        "Validation F1": round(val_score, 3)
    })

overfitting_df = pd.DataFrame(overfitting_results)

print(overfitting_df)

   Max Depth  Training F1  Validation F1
0          2        0.427          0.427
1          3        0.266          0.260
2          4        0.385          0.383
3          8        0.453          0.422
4  Unlimited        1.000          0.449


In [42]:
# Inspect the M5 preprocessor
print(preprocessor)

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['price', 'required_age']),
                                ('boolean', 'passthrough',
                                 ['windows', 'mac', 'linux']),
                                ('counts', 'passthrough',
                                 ['categories_count', 'genres_count']),
                                ('month',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['release_month']),
                                ('text',
                                 Pipeline(steps=[('clean',
                                                  FunctionTransformer(func=<function clean_text_column at 0x000002A6339631A0>)),
                                                 ('tfidf',
                                                  TfidfVectorizer(max_features=5000,
                                                                  min_df=3,
        

In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import f1_score

# Create a Random Forest model
forest_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
forest_model.fit(train_df, train_df["hit"])

# Predict validation data
forest_predictions = forest_model.predict(val_df)

# Calculate F1-score
forest_f1 = f1_score(val_df["hit"], forest_predictions)

print("Random Forest Results")
print("---------------------")
print("Validation F1-score:", round(forest_f1, 3))

Random Forest Results
---------------------
Validation F1-score: 0.021


In [44]:
from collections import Counter

print("Actual validation labels:")
print(Counter(val_df["hit"]))

print("\nRandom Forest predictions:")
print(Counter(forest_predictions))

print("\nPredicted probabilities:")
print(forest_model.predict_proba(val_df)[:, 1][:10])

Actual validation labels:
Counter({0: 14598, 1: 4908})

Random Forest predictions:
Counter({np.int64(0): 19450, np.int64(1): 56})

Predicted probabilities:
[0.23785889 0.1519763  0.21779146 0.26207959 0.2737111  0.40089407
 0.27036976 0.23481691 0.28981553 0.20730841]


In [45]:
import numpy as np
from sklearn.metrics import f1_score

# Get probabilities for the positive class (hit = 1)
forest_probabilities = forest_model.predict_proba(val_df)[:, 1]

# Try different thresholds
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("Random Forest threshold comparison")
print("----------------------------------")

for threshold in thresholds:
    predictions = (forest_probabilities >= threshold).astype(int)

    score = f1_score(val_df["hit"], predictions)

    print(
        f"Threshold: {threshold:.2f} | "
        f"F1-score: {score:.3f} | "
        f"Predicted hits: {predictions.sum()}"
    )

Random Forest threshold comparison
----------------------------------
Threshold: 0.10 | F1-score: 0.402 | Predicted hits: 19506
Threshold: 0.15 | F1-score: 0.402 | Predicted hits: 19483
Threshold: 0.20 | F1-score: 0.448 | Predicted hits: 16196
Threshold: 0.25 | F1-score: 0.537 | Predicted hits: 7786
Threshold: 0.30 | F1-score: 0.482 | Predicted hits: 3398
Threshold: 0.35 | F1-score: 0.279 | Predicted hits: 1143
Threshold: 0.40 | F1-score: 0.120 | Predicted hits: 387
Threshold: 0.45 | F1-score: 0.055 | Predicted hits: 156
Threshold: 0.50 | F1-score: 0.021 | Predicted hits: 56


In [46]:
from sklearn.metrics import classification_report

best_forest_threshold = 0.25

forest_val_predictions = (
    forest_probabilities >= best_forest_threshold
).astype(int)

print("Random Forest with threshold:", best_forest_threshold)
print("Validation F1:", round(
    f1_score(val_df["hit"], forest_val_predictions), 3
))

print("\nClassification Report:")
print(classification_report(
    val_df["hit"],
    forest_val_predictions
))

Random Forest with threshold: 0.25
Validation F1: 0.537

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.70      0.78     14598
           1       0.44      0.69      0.54      4908

    accuracy                           0.70     19506
   macro avg       0.65      0.70      0.66     19506
weighted avg       0.76      0.70      0.72     19506



In [47]:
# Combine training and validation data
development_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

print("Development dataset shape:", development_df.shape)
print("Training + validation rows:", len(development_df))
print("Test rows (kept separate):", len(test_df))

Development dataset shape: (78022, 48)
Training + validation rows: 78022
Test rows (kept separate): 19506


In [48]:
print("Development dataset columns:")
print(development_df.columns.tolist())

print("\nDataset shape:")
print(development_df.shape)

print("\nDuplicate columns:")
print(development_df.columns[development_df.columns.duplicated()].tolist())

Development dataset columns:
['appID', 'name', 'release_date', 'estimated_owners', 'peak_ccu', 'required_age', 'price', 'dlc_count', 'detailed_description', 'short_description', 'supported_languages', 'full_audio_languages', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'user_score', 'positive', 'negative', 'score_rank', 'achievements', 'recommendations', 'notes', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'developers', 'publishers', 'categories', 'genres', 'tags', 'screenshots', 'movies', 'packages', 'release_year', 'rank_in_year', 'games_in_year', 'hit', 'categories_count', 'genres_count', 'release_month']

Dataset shape:
(78022, 48)

Duplicate columns:
[]


In [49]:
from sklearn.model_selection import StratifiedKFold

# Create 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Separate features and target
X_dev = development_df.drop(columns=["hit"])
y_dev = development_df["hit"]

print("Features shape:", X_dev.shape)
print("Target shape:", y_dev.shape)

print("\nTarget distribution:")
print(y_dev.value_counts(normalize=True).round(3))

Features shape: (78022, 47)
Target shape: (78022,)

Target distribution:
hit
0    0.751
1    0.249
Name: proportion, dtype: float64


In [50]:
# Inspect the columns used by the M5 preprocessor

print("M5 preprocessing pipeline:")
print(preprocessor)

print("\nFeatures used by the preprocessor:")

for name, transformer, columns in (
    preprocessor.transformers
):
    print(f"\n{name}:")
    print(columns)

M5 preprocessing pipeline:
ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['price', 'required_age']),
                                ('boolean', 'passthrough',
                                 ['windows', 'mac', 'linux']),
                                ('counts', 'passthrough',
                                 ['categories_count', 'genres_count']),
                                ('month',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['release_month']),
                                ('text',
                                 Pipeline(steps=[('clean',
                                                  FunctionTransformer(func=<function clean_text_column at 0x000002A6339631A0>)),
                                                 ('tfidf',
                                                  TfidfVectorizer(max_features=5000,
                                                         

In [51]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression
logistic_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Decision Tree
tree_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(
        max_depth=2,
        random_state=42,
        class_weight="balanced"
    ))
])

# Random Forest
forest_cv_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

print("All three models are prepared.")

All three models are prepared.


In [52]:
from sklearn.model_selection import cross_val_score
import pandas as pd

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": forest_cv_model
}

cv_results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_dev,
        y_dev,
        cv=cv,
        scoring="f1",
        n_jobs=1
    )

    cv_results.append({
        "Model": name,
        "Mean F1": scores.mean(),
        "Standard Deviation": scores.std()
    })

    print(f"{name}:")
    print("Fold scores:", scores.round(3))
    print("Mean F1:", round(scores.mean(), 3))
    print("Standard deviation:", round(scores.std(), 3))
    print()

cv_results_df = pd.DataFrame(cv_results)

print("FINAL CROSS-VALIDATION COMPARISON")
display(cv_results_df.round(3))

Logistic Regression:
Fold scores: [0.553 0.559 0.562 0.546 0.564]
Mean F1: 0.557
Standard deviation: 0.007

Decision Tree:
Fold scores: [0.492 0.497 0.473 0.489 0.496]
Mean F1: 0.489
Standard deviation: 0.009

Random Forest:
Fold scores: [0.532 0.542 0.533 0.532 0.534]
Mean F1: 0.535
Standard deviation: 0.004

FINAL CROSS-VALIDATION COMPARISON


,Model,Mean F1,Standard Deviation
0,Logistic Regression,0.557,0.007
1,Decision Tree,0.489,0.009
2,Random Forest,0.535,0.004


In [53]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

C_values = [0.01, 0.1, 1, 10, 100]

logistic_results = []

for c in C_values:

    model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", LogisticRegression(
            C=c,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_val_score(
        model,
        X_dev,
        y_dev,
        cv=cv,
        scoring="f1",
        n_jobs=1
    )

    logistic_results.append({
        "C": c,
        "Mean F1": scores.mean(),
        "Standard Deviation": scores.std()
    })

    print(
        f"C={c}: Mean F1={scores.mean():.3f}, "
        f"Std={scores.std():.3f}"
    )

logistic_results_df = pd.DataFrame(logistic_results)

print("\nLogistic Regression tuning results:")
display(logistic_results_df.round(3))

C=0.01: Mean F1=0.544, Std=0.005
C=0.1: Mean F1=0.558, Std=0.005
C=1: Mean F1=0.557, Std=0.007
C=10: Mean F1=0.541, Std=0.006


c:\Users\kinga\anaconda3\envs\machinelearning\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\kinga\anaconda3\envs\machinelearning\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/st

C=100: Mean F1=0.535, Std=0.006

Logistic Regression tuning results:


,C,Mean F1,Standard Deviation
0,0.01,0.544,0.005
1,0.10,0.558,0.005
2,1.00,0.557,0.007
3,10.00,0.541,0.006
4,100.00,0.535,0.006


In [54]:
best_logistic_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

convergence_scores = cross_val_score(
    best_logistic_model,
    X_dev,
    y_dev,
    cv=cv,
    scoring="f1",
    n_jobs=1
)

print("Logistic Regression (C=0.1, max_iter=3000)")
print("Fold scores:", convergence_scores.round(3))
print("Mean F1:", round(convergence_scores.mean(), 3))
print("Standard deviation:", round(convergence_scores.std(), 3))

Logistic Regression (C=0.1, max_iter=3000)
Fold scores: [0.561 0.559 0.564 0.548 0.559]
Mean F1: 0.558
Standard deviation: 0.005


In [55]:
# Train the selected Logistic Regression model
best_logistic_model.fit(train_df, train_df["hit"])

# Get probabilities for validation data
logistic_probabilities = best_logistic_model.predict_proba(val_df)[:, 1]

print("Model trained successfully.")
print("Validation probabilities calculated.")

Model trained successfully.
Validation probabilities calculated.


In [56]:
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

threshold_results = []

for threshold in thresholds:

    predictions = (
        logistic_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(val_df["hit"], predictions),
        "Recall": recall_score(val_df["hit"], predictions),
        "F1": f1_score(val_df["hit"], predictions),
        "Predicted hits": predictions.sum()
    })

threshold_results_df = pd.DataFrame(threshold_results)

display(threshold_results_df.round(3))

,Threshold,Precision,Recall,F1,Predicted hits
0,0.30,0.350,0.913,0.506,12800
1,0.35,0.384,0.863,0.531,11045
2,0.40,0.415,0.806,0.548,9533
3,0.45,0.444,0.744,0.556,8219
4,0.50,0.478,0.681,0.562,6988
5,0.55,0.511,0.615,0.558,5910
6,0.60,0.548,0.551,0.550,4928
7,0.65,0.590,0.483,0.531,4017
8,0.70,0.639,0.420,0.507,3227


In [57]:
best_threshold = 0.50

print("Selected threshold:", best_threshold)
print("Validation F1:", round(
    f1_score(
        val_df["hit"],
        (logistic_probabilities >= best_threshold).astype(int)
    ), 3
))

Selected threshold: 0.5
Validation F1: 0.562


In [58]:
# Pixel Harbor can finance 10% of games
budget_percentage = 0.10

number_to_fund = int(len(val_df) * budget_percentage)

# Find the probability threshold for the top 10%
budget_threshold = pd.Series(
    logistic_probabilities
).nlargest(number_to_fund).min()

# Make predictions using this threshold
budget_predictions = (
    logistic_probabilities >= budget_threshold
).astype(int)

print("Pixel Harbor budget:", f"{budget_percentage:.0%}")
print("Maximum games to finance:", number_to_fund)
print("Calculated threshold:", round(budget_threshold, 4))

print("\nResults:")
print("Precision:", round(
    precision_score(val_df["hit"], budget_predictions), 3
))
print("Recall:", round(
    recall_score(val_df["hit"], budget_predictions), 3
))
print("F1:", round(
    f1_score(val_df["hit"], budget_predictions), 3
))
print("Recommended games:", budget_predictions.sum())

Pixel Harbor budget: 10%
Maximum games to finance: 1950
Calculated threshold: 0.7873

Results:
Precision: 0.724
Recall: 0.287
F1: 0.411
Recommended games: 1950


### Choosing the classification threshold

Pixel Harbor can only finance approximately 10% of the games it evaluates. Therefore, we chose a threshold based on the top 10% of games with the highest predicted probabilities of becoming hits.

The selected threshold was **0.7873**. At this threshold, the model recommended 1,950 out of 19,506 games. Precision was 0.724, meaning that approximately 72.4% of the recommended games were actual hits. Recall was 0.287, meaning that the model identified 28.7% of all actual hits.

The threshold that maximized F1 was 0.50, with an F1-score of 0.562. However, it recommended 6,988 games, which is far above Pixel Harbor's budget. We therefore selected 0.7873 because it better reflects the publisher's limited funding capacity, even though its F1-score is lower.

In [59]:
# Train the selected model using training + validation data
best_logistic_model.fit(
    development_df,
    development_df["hit"]
)

print("Final Logistic Regression model trained.")
print("Training games:", len(development_df))
print("Selected threshold:", best_threshold)

Final Logistic Regression model trained.
Training games: 78022
Selected threshold: 0.5


In [60]:
# Use the threshold that matches Pixel Harbor's 10% budget
best_threshold = budget_threshold

print("Final selected threshold:", round(best_threshold, 4))

Final selected threshold: 0.7873


In [61]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Predict probabilities for the test set
test_probabilities = best_logistic_model.predict_proba(test_df)[:, 1]

# Apply the selected threshold
test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

# Evaluate the model
print("FINAL TEST RESULTS")
print("------------------")
print("Threshold:", round(best_threshold, 4))
print("Accuracy:", round(accuracy_score(test_df["hit"], test_predictions), 3))
print("Precision:", round(precision_score(test_df["hit"], test_predictions), 3))
print("Recall:", round(recall_score(test_df["hit"], test_predictions), 3))
print("F1-score:", round(f1_score(test_df["hit"], test_predictions), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(test_df["hit"], test_predictions))

print("\nClassification Report:")
print(classification_report(test_df["hit"], test_predictions))

FINAL TEST RESULTS
------------------
Threshold: 0.7873
Accuracy: 0.796
Precision: 0.741
Recall: 0.301
F1-score: 0.428

Confusion Matrix:
[[14030   522]
 [ 3462  1492]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.96      0.88     14552
           1       0.74      0.30      0.43      4954

    accuracy                           0.80     19506
   macro avg       0.77      0.63      0.65     19506
weighted avg       0.79      0.80      0.76     19506



### Final test evaluation

The selected model was Logistic Regression with C=0.1 and a classification threshold of 0.7873. The model was trained using 78,022 games from the training and validation datasets.

On the final test dataset of 19,506 games, the model achieved an accuracy of 0.796, precision of 0.741, recall of 0.301 and F1-score of 0.428.

The results show that the model is relatively good at identifying games that are likely to become hits. However, its recall is lower, meaning that it misses many games that actually become successful. This is a trade-off that makes sense for Pixel Harbor because it has a limited budget and wants to prioritize games with a higher probability of success.

In [62]:
print("Games by release year in development data:")
print(development_df["release_year"].value_counts().sort_index())

print("\nGames up to 2022:")
print((development_df["release_year"] <= 2022).sum())

print("\nGames from 2023–2024:")
print(
    development_df["release_year"].between(2023, 2024).sum()
)

Games by release year in development data:
release_year
2006       48
2007       75
2008      127
2009      260
2010      207
2011      208
2012      254
2013      358
2014     1199
2015     1983
2016     3338
2017     4770
2018     5974
2019     5804
2020     7027
2021     8859
2022     9850
2023    11625
2024    16056
Name: count, dtype: int64

Games up to 2022:
50341

Games from 2023–2024:
27681


In [63]:
# Split development data by release year
past_df = development_df[
    development_df["release_year"] <= 2022
].copy()

future_df = development_df[
    development_df["release_year"].between(2023, 2024)
].copy()

# Separate features and target
X_past = past_df.drop(columns=["hit"])
y_past = past_df["hit"]

X_future = future_df.drop(columns=["hit"])
y_future = future_df["hit"]

print("Past training data:", X_past.shape)
print("Future test data:", X_future.shape)

print("\nPast target distribution:")
print(y_past.value_counts(normalize=True).round(3))

print("\nFuture target distribution:")
print(y_future.value_counts(normalize=True).round(3))

Past training data: (50341, 47)
Future test data: (27681, 47)

Past target distribution:
hit
0    0.751
1    0.249
Name: proportion, dtype: float64

Future target distribution:
hit
0    0.751
1    0.249
Name: proportion, dtype: float64


In [64]:
# Create a fresh model with the selected settings
future_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train using only games released up to 2022
future_model.fit(X_past, y_past)

print("Future-data model trained successfully.")
print("Training games:", len(X_past))
print("Future test games:", len(X_future))

Future-data model trained successfully.
Training games: 50341
Future test games: 27681


In [65]:
# Predict probabilities for future games
future_probabilities = future_model.predict_proba(X_future)[:, 1]

# Apply the same threshold selected for Pixel Harbor
future_predictions = (
    future_probabilities >= best_threshold
).astype(int)

# Calculate evaluation metrics
print("FUTURE-DATA TEST RESULTS")
print("------------------------")
print("Threshold:", round(best_threshold, 4))
print("Accuracy:", round(accuracy_score(y_future, future_predictions), 3))
print("Precision:", round(precision_score(y_future, future_predictions), 3))
print("Recall:", round(recall_score(y_future, future_predictions), 3))
print("F1-score:", round(f1_score(y_future, future_predictions), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_future, future_predictions))

print("\nClassification Report:")
print(classification_report(y_future, future_predictions))

FUTURE-DATA TEST RESULTS
------------------------
Threshold: 0.7873
Accuracy: 0.775
Precision: 0.668
Recall: 0.189
F1-score: 0.294

Confusion Matrix:
[[20137   648]
 [ 5594  1302]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.97      0.87     20785
           1       0.67      0.19      0.29      6896

    accuracy                           0.77     27681
   macro avg       0.73      0.58      0.58     27681
weighted avg       0.75      0.77      0.72     27681



### Future-data evaluation

To evaluate how well the model performs on newer games, a separate temporal experiment was conducted. The model was trained on 50,341 games released between 2006 and 2022 and evaluated on 27,681 games released in 2023 and 2024.

The same classification threshold of 0.7873 was used. The model achieved an accuracy of 0.775, precision of 0.668, recall of 0.189 and F1-score of 0.294.

Compared with the original test results, performance decreased. The F1-score dropped from 0.428 to 0.294, while recall decreased from 0.301 to 0.189. This suggests that the model has more difficulty identifying successful games from newer release years when trained only on older games.

One possible explanation is that the characteristics associated with successful games change over time. Therefore, the model may need to be updated regularly with more recent game data.

In [66]:
import joblib
from pathlib import Path

# Create a folder for saved models
model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

# Save model and threshold together
model_artifact = {
    "model": best_logistic_model,
    "threshold": best_threshold,
    "model_name": "Logistic Regression",
    "C": 0.1,
    "max_iter": 3000
}

model_path = model_dir / "steam_hit_model.joblib"

joblib.dump(model_artifact, model_path)

print("Model saved successfully!")
print("Location:", model_path)
print("Threshold:", best_threshold)

Model saved successfully!
Location: ..\models\steam_hit_model.joblib
Threshold: 0.7873065163529368


In [67]:
import joblib

# Load the saved model
loaded_artifact = joblib.load("../models/steam_hit_model.joblib")

loaded_model = loaded_artifact["model"]
loaded_threshold = loaded_artifact["threshold"]

# Make a test prediction using one game
sample_game = test_df.drop(columns=["hit"]).iloc[[0]]

sample_probability = loaded_model.predict_proba(sample_game)[0, 1]

sample_prediction = (
    sample_probability >= loaded_threshold
).astype(int)

print("Model loaded successfully!")
print("Threshold:", round(loaded_threshold, 4))
print("Sample probability:", round(sample_probability, 4))
print("Prediction:", "HIT" if sample_prediction else "NOT A HIT")

Model loaded successfully!
Threshold: 0.7873
Sample probability: 0.2717
Prediction: NOT A HIT
